<a href="https://colab.research.google.com/github/mohanasudhashanmugam/DeepLearning/blob/main/DL_Objdetection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

To download and unzip the datasets

In [2]:
!kaggle datasets download -d kipshidze/shoplifting-video-dataset


Dataset URL: https://www.kaggle.com/datasets/kipshidze/shoplifting-video-dataset
License(s): Attribution 4.0 International (CC BY 4.0)
100% 726M/726M [00:13<00:00, 56.7MB/s]



In [3]:
!unzip -q shoplifting-video-dataset.zip -d ./local_colab_storage

Dataset path Extraction



In [4]:
!ls /content/local_colab_storage/normal | wc -l

90


In [5]:
!ls /content/local_colab_storage/shoplifting | wc -l

92


In [24]:
import cv2
import os
import numpy as np
import argparse
from imutils import paths
from tensorflow.keras.utils import to_categorical
from sklearn.preprocessing import LabelBinarizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

In [7]:
# construct the argument parser and parse the arguments
ap = argparse.ArgumentParser()

ap.add_argument("-d", "--dataset",
	default="/content/local_colab_storage/",
	help="path to input dataset")
args_namespace, unknown = ap.parse_known_args()


# Convert to a dictionary
args = vars(args_namespace)


Frame extraction

In [35]:
def process_video(video_path, max_frames=16, resize_dim=(224, 224)):
    """
    Opens a video, uniformly extracts a fixed number of frames,
    resizes them, and normalizes pixel values.
    """
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    # Handle empty or corrupted videos
    if total_frames <= 0:
        cap.release()
        return None

    # Calculate uniform intervals to pick frames across the whole video duration
    # This ensures a 5-second video and a 20-second video both yield exactly 'max_frames'
    frame_indices = np.linspace(0, total_frames - 1, max_frames, dtype=int)

    video_frames = []

    for frame_idx in frame_indices:
        # Set the reader to the specific frame index
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
        success, frame = cap.read()

        if not success:
            break

        # 1. Convert color from BGR (OpenCV default) to RGB
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        # 2. Resize the frame (e.g., to 224x224)
        frame_resized = cv2.resize(frame, resize_dim)

        # 3. Normalize pixel values by dividing by 255.0 (converts 0-255 integers to 0.0-1.0 floats)
        frame_normalized = frame_resized / 255.0

        video_frames.append(frame_normalized)

    cap.release()

    # If the video didn't have enough readable frames, pad it or skip it
    if len(video_frames) < max_frames:
        return None

    # Convert list of frames into a single NumPy array
    # Output shape: (16, 224, 224, 3)
    return np.array(video_frames, dtype=np.float32)

# --- EXAMPLE USAGE ON ONE FILE ---
# (Replace with your actual unzipped path from !ls)
## sample_path = "./local_colab_storage/"


# 1. Define the video extensions you want to look for
video_extensions = (".mp4", ".avi", ".mkv", ".mov", ".wmv", ".flv", ".webm")

# 2. Grab all matching video paths recursively
video_paths = list(paths.list_files(args["dataset"], validExts=video_extensions))
print("video_paths: ", video_paths)

data=[]
labels=[]

#if os.path.exists(video_paths):
for video_path in video_paths:

  label = video_path.split(os.path.sep)[-2]
  labels.append(label)

  processed_tensor = process_video(video_path, max_frames=16, resize_dim=(224, 224))
  if processed_tensor is not None:
    data.append(processed_tensor)

  print("Video Processed Successfully!")
  print(f"Final Tensor Shape: {processed_tensor.shape}") # Expecting (16, 224, 224, 3)
  print(f"Min pixel value: {processed_tensor.min()}, Max pixel value: {processed_tensor.max()}")





video_paths:  ['/content/local_colab_storage/shoplifting/shoplifting-4.mp4', '/content/local_colab_storage/shoplifting/shoplifting-59.mp4', '/content/local_colab_storage/shoplifting/shoplifting-19.mp4', '/content/local_colab_storage/shoplifting/shoplifting-39.mp4', '/content/local_colab_storage/shoplifting/shoplifting-31.mp4', '/content/local_colab_storage/shoplifting/shoplifting-67.mp4', '/content/local_colab_storage/shoplifting/shoplifting-72.mp4', '/content/local_colab_storage/shoplifting/shoplifting-80.mp4', '/content/local_colab_storage/shoplifting/shoplifting-42.mp4', '/content/local_colab_storage/shoplifting/shoplifting-83.mp4', '/content/local_colab_storage/shoplifting/shoplifting-11.mp4', '/content/local_colab_storage/shoplifting/shoplifting-41.mp4', '/content/local_colab_storage/shoplifting/shoplifting-48.mp4', '/content/local_colab_storage/shoplifting/shoplifting-61.mp4', '/content/local_colab_storage/shoplifting/shoplifting-71.mp4', '/content/local_colab_storage/shoplifting

In [30]:
##labels = np.array(labels)

In [36]:
# perform one-hot encoding on the labels
lb = LabelBinarizer()
labels = lb.fit_transform(labels)
labels = to_categorical(labels)

In [38]:
# Convert lists to final NumPy arrays
X = np.array(data, dtype=np.float32)
y = np.array(labels, dtype=np.int32)
print(f"Data loading complete!")
print(f"X shape (Videos, Frames, H, W, Channels): {X.shape}")
print(f"y shape (Labels): {y.shape}")


Data loading complete!
X shape (Videos, Frames, H, W, Channels): (182, 16, 224, 224, 3)
y shape (Labels): (182, 2)


Split into Train and Test Sets

In [39]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training samples: {X_train.shape[0]} | Testing samples: {X_test.shape[0]}")

Training samples: 145 | Testing samples: 37


CNN + LSTM

The CNN (MobileNetV2): Looks at each individual frame and extracts spatial features (like a hand, a bag, or a shelf).
The LSTM: Looks at how those spatial features change over the 16 frames to understand the action (the movement of hiding an item).

In [42]:
import tensorflow as tf
from tensorflow.keras import layers, models

# 1. Base CNN to extract features from a single frame
# We use MobileNetV2 because it is lightweight and fast
base_cnn = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3), include_top=False, weights='imagenet'
)

base_cnn.trainable = False  # Freeze weights as we don't destroy pre-trained weights by new training data during backpropagation


# Flatten the CNN output to a vector
pooling_layer = layers.GlobalAveragePooling2D()(base_cnn.output)
feature_extractor = models.Model(inputs=base_cnn.input, outputs=pooling_layer)

# 2. Complete Video Model
video_input = layers.Input(shape=(16, 224, 224, 3)) # (Frames, Height, Width, Channels)

# TimeDistributed applies the CNN to all 16 frames individually
encoded_frames = layers.TimeDistributed(feature_extractor)(video_input)

# LSTM tracks the movement across the timeline
x = layers.LSTM(64, dropout=0.5)(encoded_frames)
x = layers.Dense(32, activation='relu')(x)

# Output layer: Sigmoid activation for binary classification (0 or 1)
output = layers.Dense(2, activation='sigmoid')(x)

model = models.Model(inputs=video_input, outputs=output)
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

model.summary()


Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)      │ (None, 16, 224, 224,   │             0 │
│                                 │ 3)                     │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_1              │ (None, 16, 1280)       │     2,257,984 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 64)             │       344,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 2)              │            66 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,604,450 (9.94 MB)

 Trainable params: 346,466 (1.32 MB)

 Non-trainable params: 2,257,984 (8.61 MB)

 Training Model by feeding processed arrays into the model

In [43]:

history = model.fit(
    X_train, y_train,
    validation_split=0.1, # Uses a slice of training data to check performance mid-training
    epochs=10,
    batch_size=8 # Small batch size keeps memory safe
)

Epoch 1/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 86s 2s/step - accuracy: 0.5538 - loss: 0.7034 - val_accuracy: 0.8667 - val_loss: 0.6600
Epoch 2/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 5s 284ms/step - accuracy: 0.5154 - loss: 0.6938 - val_accuracy: 0.8000 - val_loss: 0.6421
Epoch 3/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 5s 309ms/step - accuracy: 0.5769 - loss: 0.6716 - val_accuracy: 0.7333 - val_loss: 0.6443
Epoch 4/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 5s 288ms/step - accuracy: 0.7077 - loss: 0.6163 - val_accuracy: 0.8000 - val_loss: 0.5956
Epoch 5/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 5s 294ms/step - accuracy: 0.6923 - loss: 0.5875 - val_accuracy: 0.4000 - val_loss: 0.7545
Epoch 6/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 5s 311ms/step - accuracy: 0.6154 - loss: 0.6342 - val_accuracy: 0.8667 - val_loss: 0.4790
Epoch 7/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 10s 308ms/step - accuracy: 0.7538 - loss: 0.5314 - val_accuracy: 0.8667 - val_loss: 0.4814
Epoch 8/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 5s 292ms/step - accuracy: 0.7462 - loss: 0.5228 - val_accuracy: 0.8

In [56]:
from sklearn.metrics import classification_report, confusion_matrix

# 1. Get raw probability predictions (numbers between 0.0 and 1.0)
predictions = model.predict(X_test)
print(predictions)


2/2 ━━━━━━━━━━━━━━━━━━━━ 2s 296ms/step
[[0.80777216 0.19048378]
 [0.9310414  0.06539708]
 [0.89538074 0.11968543]
 [0.4426131  0.50563973]
 [0.8135101  0.18922101]
 [0.7488209  0.2529588 ]
 [0.56982404 0.4211266 ]
 [0.62007046 0.41935948]
 [0.6102782  0.34730133]
 [0.60145503 0.39378166]
 [0.8230582  0.177686  ]
 [0.8611081  0.12946253]
 [0.8667436  0.11553327]
 [0.86175513 0.14341207]
 [0.6008884  0.38936228]
 [0.51587725 0.41543055]
 [0.74232763 0.22003284]
 [0.9208449  0.13199355]
 [0.7725366  0.20395814]
 [0.8606365  0.14889333]
 [0.80553997 0.22251524]
 [0.45451894 0.5203887 ]
 [0.52547073 0.47569504]
 [0.9345623  0.06425947]
 [0.8232718  0.18438652]
 [0.49979338 0.4503207 ]
 [0.61075854 0.39698794]
 [0.6482941  0.37104344]
 [0.9395472  0.05995911]
 [0.18833049 0.78719324]
 [0.5016158  0.480827  ]
 [0.6740741  0.3020007 ]
 [0.89546484 0.11428656]
 [0.5586192  0.4607542 ]
 [0.9053723  0.11770472]
 [0.40707657 0.5881312 ]
 [0.18822056 0.7447381 ]]


In [54]:
print(y_test)

[[0 1]
 [1 0]
 [1 0]
 [0 1]
 [0 1]
 [1 0]
 [0 1]
 [0 1]
 [0 1]
 [1 0]
 [0 1]
 [1 0]
 [0 1]
 [1 0]
 [0 1]
 [0 1]
 [1 0]
 [1 0]
 [0 1]
 [1 0]
 [1 0]
 [1 0]
 [1 0]
 [1 0]
 [0 1]
 [0 1]
 [0 1]
 [0 1]
 [1 0]
 [0 1]
 [1 0]
 [1 0]
 [1 0]
 [0 1]
 [1 0]
 [0 1]
 [0 1]]


In [59]:
# 2. Convert probabilities to binary choices: if > 0.5, it's Shoplifting (1), else Normal (0)
##binary_predictions = (predictions > 0.5).astype(int)

binary_predictions = np.argmax(predictions, axis=1)

# Convert y_test from 2 columns to 1 column
y_test_labels = np.argmax(y_test, axis=1)

print(binary_predictions)

[0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 1 0 0 0 0 0 1 1]


In [60]:
print("--- Confusion Matrix ---")
print(confusion_matrix(y_test_labels, binary_predictions))



--- Confusion Matrix ---
[[17  1]
 [15  4]]


In [63]:
print("\n--- Classification Report ---")
print(classification_report(y_test_labels, binary_predictions, target_names=['Normal', 'Shoplifting']))


--- Classification Report ---
              precision    recall  f1-score   support

      Normal       0.53      0.94      0.68        18
 Shoplifting       0.80      0.21      0.33        19

    accuracy                           0.57        37
   macro avg       0.67      0.58      0.51        37
weighted avg       0.67      0.57      0.50        37

